In [11]:
from torch import nn
import torch
from torchvision import transforms,datasets
from torch.utils.data import DataLoader

In [12]:
device=torch.device( "cuda" if torch.cuda.is_available() else "cpu")

transform=transforms.Compose([transforms.ToTensor()])
train_dataset=datasets.MNIST(transform=transform,download=True,train=True,root="data")
test_dataset=datasets.MNIST(transform=transform,download=True,train=False,root="data")
train_loader=DataLoader(train_dataset,shuffle=True,batch_size=32)
test_loader=DataLoader(test_dataset,shuffle=False,batch_size=32)

In [13]:
class GRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru=nn.GRU(input_size=28,hidden_size=64,num_layers=2,batch_first=True)
        self.fc=nn.Linear(64,10)
    def forward(self,x):
        output,hidden=self.gru(x)
        x=output[:,-1,:]
        x=self.fc(x)
        return x

In [19]:
epoch=5
model=GRU()
model.to(device)
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
loss_func=nn.CrossEntropyLoss()
for i in range(epoch):
    model.train()
    total_loss=0
    for data,actual_result in train_loader:
        data=data.squeeze(1)
        data=data.to(device)
        optimizer.zero_grad()
        actual_result=actual_result.to(device)
        predicted=model(data)
        loss=loss_func(predicted,actual_result)
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    model.eval()
    correct=0
    total=0
    for data,actual_result in test_loader:
        data=data.squeeze(1)
        data=data.to(device)
        actual_result=actual_result.to(device)
        predicted=model(data)
        correct+=(predicted.argmax(dim=1)==actual_result).sum().item()
        total+=actual_result.size(0)
    print(f"loss: {total_loss/len(train_loader)} validation accuracy {correct/total}")

loss: 0.40473622331122555 validation accuracy 0.9617
loss: 0.11733779831727346 validation accuracy 0.9624
loss: 0.07632280904675523 validation accuracy 0.9788
loss: 0.057361057648869854 validation accuracy 0.9807
loss: 0.047080148255452516 validation accuracy 0.9854
